# SYMFLUENCE tutorial 04a - Logan River workshop (distributed ASYNC-DDS)

## Introduction

This workshop notebook demonstrates semidistributed NGEN hydrological modelling
using SYMFLUENCE's distributed Logan River setup with cloud-based RDRS forcing.
The domain definition follows `04a_logan_river_workshop_distributed.ipynb`, while
the model chain is set up to be closer to the local NGIAB realization: SLOTH,
NOAH-OWP, CFE, and t-route.

The workflow includes:

1. **Configuration** - Set up a semidistributed NGEN model for the Logan River
2. **Domain definition** - Delineate subcatchments using TauDEM
3. **Data acquisition** - Fetch RDRS forcing data and USGS streamflow observations
4. **Model execution** - Run NGEN with SLOTH, NOAH-OWP, CFE, and t-route
5. **Evaluation and calibration** - Assess model performance and calibrate parameters

The **Logan River at Logan** is a snow-dominated mountain watershed in the Bear
River Range of the Wasatch Mountains. USGS station 10109000 provides streamflow
observations. The distributed setup preserves subcatchment-scale routing instead
of collapsing the watershed into a single lumped response unit.


In [ ]:
# Environment verification
import sys
import os
import importlib
import warnings
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

# Suppress experimental module warnings for cleaner output
warnings.filterwarnings('ignore', message='.*is an EXPERIMENTAL module.*')
warnings.filterwarnings('ignore', message='.*import failed.*')

print(f"Python executable: {sys.executable}")

# Verify SYMFLUENCE is available
try:
    import symfluence
    from symfluence.models.ngen import preprocessor as ngen_preprocessor
    from symfluence.models.ngen.calibration import worker as ngen_worker

    importlib.reload(ngen_preprocessor)
    importlib.reload(ngen_worker)
    print(f"SYMFLUENCE version: {symfluence.__version__}")
    print(f"SYMFLUENCE location: {Path(symfluence.__file__).parent}")
except ImportError:
    print("ERROR: SYMFLUENCE not found. Please activate the symfluence environment.")
    sys.exit(1)


In [ ]:
# Fix working directory if running from .ipynb_checkpoints
current_dir = Path.cwd()
print(f"Current directory: {current_dir}")

# If we're in .ipynb_checkpoints, move up to parent directory
if '.ipynb_checkpoints' in str(current_dir):
    correct_dir = current_dir.parent
    os.chdir(correct_dir)
    print(f"Changed to: {Path.cwd()}")
else:
    print("Working directory is correct")

# Verify we're in the workshop notebooks directory
expected_notebook = Path.cwd() / '04a_logan_river_workshop_asyncdds_distributed_ngiab.ipynb'
if not expected_notebook.exists():
    print(f"WARNING: Expected notebook not found at {expected_notebook}")
else:
    print("Notebook location verified")


## Step 1 - Configuration

Create a configuration for the Logan River semidistributed NGEN model. Key
settings:
- **Domain**: TauDEM semidistributed setup from the distributed workshop notebook
- **Forcing**: RDRS hourly gridded data, unchanged from the workshop
- **Observations**: USGS streamflow from station 10109000
- **Period**: 4 years (2018-2021) with 1-year spinup


In [ ]:
# Step 1 - Create distributed NGEN configuration
import symfluence as symfluence_pkg
from symfluence import SYMFLUENCE
from symfluence.core.config.models import SymfluenceConfig


def resolve_symfluence_paths():
    """Resolve SYMFLUENCE code and data paths for local and JupyterHub installs."""
    pkg_dir = Path(symfluence_pkg.__file__).resolve().parent
    repo_root = None

    for candidate in [pkg_dir, *pkg_dir.parents]:
        if (
            (candidate / "src" / "symfluence").exists()
            or (candidate / ".git").exists()
            or (candidate / "pyproject.toml").exists()
        ):
            repo_root = candidate.resolve()
            break

    if repo_root is not None:
        code_dir = repo_root
        data_dir = repo_root.parent / "SYMFLUENCE_data"
        mode = "symfluence repository detected"
    else:
        code_dir = Path(
            os.environ.get("SYMFLUENCE_CODE_DIR", str(Path.home() / "SYMFLUENCE"))
        )
        data_dir = Path(
            os.environ.get(
                "SYMFLUENCE_DATA_DIR",
                str(Path.home() / "SYMFLUENCE_data"),
            )
        )
        mode = "installed package / JupyterHub fallback"

    return code_dir, data_dir, mode


code_dir, data_dir, path_mode = resolve_symfluence_paths()

if not data_dir.exists():
    raise RuntimeError(
        f"SYMFLUENCE data directory does not exist: {data_dir}\n"
        "Set SYMFLUENCE_DATA_DIR or create the directory before continuing."
    )

installs_dir = data_dir / "installs"
if not installs_dir.exists():
    print(
        f"WARNING: installs directory not found at {installs_dir}.\n"
        "If using the JupyterHub/Docker image, ensure your installs symlink is configured."
    )

print(f"Notebook directory: {current_dir}")
print(f"SYMFLUENCE path resolution mode: {path_mode}")
print(f"SYMFLUENCE data directory: {data_dir}")
print(f"SYMFLUENCE code directory: {code_dir}")
print(f"SYMFLUENCE installs directory: {installs_dir}")

# Use a conservative local worker count for NGEN calibration
calibration_workers = 2
calibration_evaluations = 20
print(f"Calibration workers: {calibration_workers}")
print(f"Target calibration evaluations: {calibration_evaluations}")

# Build config kwargs
config_kwargs = dict(
    # Basic identification
    domain_name="Logan_River_at_Logan",
    experiment_id="workshop_run_distributed_asyncdds",

    # Paths
    SYMFLUENCE_DATA_DIR=str(data_dir),
    SYMFLUENCE_CODE_DIR=str(code_dir),
    TAUDEM_DIR="default",

    # Model - NGEN with SLOTH + NOAH-OWP + CFE and t-route
    model="NGEN",
    routing_model="none",
    NGEN_MODULES_SELECTED="SLOTH,NOAH,CFE",
    NGEN_MODULES_TO_CALIBRATE="CFE,NOAH",
    NGEN_CFE_PARAMS_TO_CALIBRATE="maxsmc,satdk,bb,slop",
    NGEN_NOAH_PARAMS_TO_CALIBRATE="refkdt,slope,smcmax,dksat",
    NGEN_PET_PARAMS_TO_CALIBRATE="",
    # CFE receives Noah evapotranspiration, matching the local NGIAB realization
    NGEN_NOAH_ET_FALLBACK="EVAPOTRANS",
    NGEN_RUN_TROUTE=True,
    NUM_PROCESSES=calibration_workers,

    # Simulation period (4 years: 2018-2021)
    time_start="2018-01-01 01:00",
    time_end="2021-12-31 23:00",
    spinup_period="2018-01-01, 2018-12-31",
    calibration_period="2019-01-01, 2019-12-31",
    evaluation_period="2020-01-01, 2020-12-31",

    # Spatial domain from the distributed workshop notebook
    pour_point_coords="41.743098/-111.786432",
    bounding_box_coords="42.15/-111.90/41.70/-111.40",
    definition_method="semi-distributed",
    discretization="GRUs",
    stream_threshold=5000,
    lumped_watershed_method="TauDEM",

    # Data sources
    data_access="cloud",
    forcing_dataset="RDRS",
    forcing_measurement_height=2,
    dem_source="copdem90",
    download_dem=True,

    # Streamflow observations
    station_id="10109000",
    streamflow_data_provider="USGS",
    download_usgs_data=True,

    # Calibration
    OPTIMIZATION_METHODS=["iteration"],
    optimization_target="streamflow",
    iterative_optimization_algorithm="ASYNC-DDS",
    optimization_metric="KGE",
    calibration_timestep="hourly",
    iterations=max(1, calibration_evaluations // calibration_workers),
    SKIP_WARM_START=True,
    ASYNC_DDS_POOL_SIZE=calibration_workers * 2,
    ASYNC_DDS_BATCH_SIZE=calibration_workers,
)
config = SymfluenceConfig.from_minimal(**config_kwargs)

# Save configuration
config_path = Path("./config_logan_river_distributed_asyncdds.yaml")
config_dict = config.to_dict(flatten=True)
import yaml
with open(config_path, "w") as f:
    yaml.dump(config_dict, f, default_flow_style=False, sort_keys=False)
print(f"  Config saved to: {config_path}")

# Initialize SYMFLUENCE
symfluence = SYMFLUENCE(config)
project_dir = symfluence.managers["project"].setup_project()
pour_point_path = symfluence.managers["project"].create_pour_point()

print(f"\nProject structure created at: {project_dir}")
print(f"Pour point shapefile: {pour_point_path}")
print("=" * 80)


## Step 2 - Domain definition

Delineate the Logan River watershed using the TauDEM semidistributed setup from
`04a_logan_river_workshop_distributed.ipynb`.


### Step 2a — Geospatial Attribute Acquisition

Acquire elevation, land cover, and soil data from cloud sources.

In [ ]:
# Step 2a — Acquire geospatial attributes from cloud
symfluence.managers['data'].acquire_attributes()
print("Attribute acquisition complete")

### Step 2b — Watershed Delineation

Delineate the watershed boundary from the pour point using TauDEM.

In [ ]:
# Step 2b — Watershed delineation
watershed_path = symfluence.managers['domain'].define_domain()
print(f"Watershed delineation complete")

### Step 2c - Domain discretization

Create GRUs from the semidistributed watershed.


In [ ]:
# Step 2c — Discretization 
hru_path = symfluence.managers['domain'].discretize_domain()
print("Domain discretization complete")

### Step 2d — Visualization

Visualize the delineated watershed and pour point.

In [ ]:
# Step 2d - Basin visualization
# Load spatial data from the completed domain and discretization workflows
domain_artifacts = symfluence.managers["domain"].delineation_artifacts
discretization_artifacts = symfluence.managers["domain"].discretization_artifacts
basin_path = domain_artifacts.river_basins_path
hru_file = discretization_artifacts.hru_paths
if isinstance(hru_file, dict):
    hru_file = next(iter(hru_file.values()))

watershed_gdf = gpd.read_file(str(basin_path))
hru_gdf = gpd.read_file(str(hru_file))
pour_point_gdf = gpd.read_file(pour_point_path)

# Calculate area (UTM Zone 12N for Utah)
watershed_proj = watershed_gdf.to_crs("EPSG:32612")
area_km2 = watershed_proj.geometry.area.sum() / 1e6

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
watershed_gdf.boundary.plot(ax=ax, color="blue", linewidth=1.0, label="Watershed")
hru_gdf.plot(ax=ax, facecolor="lightblue", edgecolor="blue", alpha=0.3)
pour_point_gdf.plot(
    ax=ax,
    color="red",
    markersize=150,
    marker="*",
    label=f"Pour Point (USGS {config.evaluation.streamflow.station_id})",
)

ax.set_title(
    f"Logan River at Logan\nArea: {area_km2:.0f} km²",
    fontweight="bold",
    fontsize=14,
)
ax.legend(loc="upper right")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

print(f"Watershed area: {area_km2:.0f} km²")
print(f"Number of GRUs: {len(hru_gdf)}")
print(f"Number of subcatchments: {len(watershed_gdf)}")


## Step 3 — Data Acquisition and Preprocessing

Fetch forcing data (RDRS) and streamflow observations (USGS) from cloud sources.

### Step 3a — USGS Streamflow Observations

Download and process USGS streamflow data for station 10109000.

In [ ]:
# Step 3a — Download and process USGS streamflow data
symfluence.managers['data'].process_observed_data()                                                                                                      
print("USGS streamflow data acquisition complete")

### Step 3b — RDRS Meteorological Forcing

Download RDRS forcing data from cloud. RDRS (Regional Deterministic Reanalysis System) provides:
- Hourly temporal resolution
- Complete forcing variables: precipitation, temperature, humidity, wind, radiation, pressure

In [ ]:
# Step 3b — Acquire RDRS forcing data from cloud
symfluence.managers['data'].acquire_forcings()
print("RDRS forcing acquisition complete")

### Step 3c - Model-agnostic preprocessing

Standardize forcing data and remap it to the current GRUs. If the remapping
cache was created for a different domain discretization, clear only those
derived files before rebuilding.


In [ ]:
# Step 3c - Model-agnostic preprocessing
import shutil
import xarray as xr

# Remove stale forcing remapping outputs before rebuilding model-ready forcing
discretization_artifacts = symfluence.managers["domain"].discretization_artifacts
if discretization_artifacts is None or discretization_artifacts.hru_paths is None:
    raise RuntimeError("Run Step 2c before forcing preprocessing.")

hru_file = discretization_artifacts.hru_paths
if isinstance(hru_file, dict):
    hru_file = next(iter(hru_file.values()))

current_hru_count = len(gpd.read_file(hru_file))
forcing_cache_dir = project_dir / "data" / "forcing" / "basin_averaged_data"
forcing_intersection_dir = (
    project_dir / "shapefiles" / "catchment_intersection" / "with_forcing"
)
remapped_files = sorted(forcing_cache_dir.glob("*.nc")) if forcing_cache_dir.exists() else []

cached_hru_count = None
if remapped_files:
    with xr.open_dataset(remapped_files[0]) as remapped_ds:
        cached_hru_count = remapped_ds.sizes.get("hru")

stale_remap = cached_hru_count is not None and cached_hru_count != current_hru_count
orphaned_weights = not remapped_files and forcing_intersection_dir.exists()

if stale_remap:
    print(
        "Removing stale forcing remap cache: "
        f"cached hru={cached_hru_count}, current hru={current_hru_count}"
    )
elif orphaned_weights:
    print("Removing orphaned forcing remap weights without remapped forcing files")
else:
    print(f"Forcing remap cache is ready for {current_hru_count} HRUs")

if stale_remap or orphaned_weights:
    for stale_path in [forcing_cache_dir, forcing_intersection_dir]:
        if stale_path.exists():
            shutil.rmtree(stale_path)
            print(f"Removed stale forcing remap path: {stale_path}")

symfluence.managers["data"].run_model_agnostic_preprocessing()
print("Model-agnostic preprocessing complete")


## Step 4 — Model Configuration and Execution

Run model-specific preprocessing and execute the selected model.

In [ ]:
# Step 4a — Model-specific preprocessing
print(f"Preprocessing for {config.model.hydrological_model}...")
symfluence.managers['model'].preprocess_models()
print(f"{config.model.hydrological_model} preprocessing complete")

In [ ]:
# Step 4b — Model execution
print(f"Running {config.model.hydrological_model}...")
print(f"Simulation period: {config.domain.time_start} to {config.domain.time_end}")
symfluence.managers['model'].run_models()
print("Model execution complete")

# Step 4c — Post-processing (extract streamflow to standardized results CSV)
print(f"Post-processing for {config.model.hydrological_model}")
symfluence.managers['model'].postprocess_results()
print("Post-processing complete")

## Step 5 — Streamflow Evaluation

Compare simulated streamflow against USGS observations using standard hydrological metrics.

In [ ]:
model_name = config.model.hydrological_model
experiment_id = config.domain.experiment_id

results_file = project_dir / "results" / f"{experiment_id}_results.csv"
results_df = pd.read_csv(results_file, index_col=0, parse_dates=True)
print(f"Loaded results: {results_file}")
print(f"Columns: {list(results_df.columns)}")

# Find simulation column (any *_discharge_cms column)
sim_cols = [c for c in results_df.columns if 'discharge' in c.lower() and 'obs' not in c.lower()]
if not sim_cols:
    raise ValueError(f"No discharge column found. Available: {list(results_df.columns)}")
sim_col = sim_cols[0]
print(f"Using simulation column: {sim_col}")

# Load observed streamflow
obs_dir = project_dir / "data" / "observations" / "streamflow" / "preprocessed"
obs_path = obs_dir / f"{config.domain.name}_streamflow_processed.csv"
if obs_path.exists():
    obs_df = pd.read_csv(obs_path)
    obs_df['datetime'] = pd.to_datetime(obs_df['datetime'], utc=True, errors='coerce').dt.tz_convert(None)
    obs_df.set_index('datetime', inplace=True)
    obs_daily = obs_df['discharge_cms'].resample('D').mean()
else:
    print(f"Warning: observations not found at {obs_path}")
    obs_daily = None

# Align simulation and observations
sim_series = results_df[sim_col].resample('D').mean()

# Exclude spinup period
spinup_end = pd.to_datetime(config.domain.spinup_period.split(',')[1].strip())
sim_series = sim_series[sim_series.index > spinup_end]

common_idx = sim_series.index.intersection(obs_daily.index)
obs_valid = obs_daily.loc[common_idx].dropna()
sim_valid = sim_series.loc[obs_valid.index]

# Remove NaN pairs
mask = ~(np.isnan(obs_valid.values) | np.isnan(sim_valid.values))
obs_clean = obs_valid[mask]
sim_clean = sim_valid[mask]

print(f"\nExcluding spinup up to: {spinup_end}")
print(f"Evaluation period: {obs_clean.index[0]} to {obs_clean.index[-1]}")
print(f"Valid data points: {len(obs_clean)}")

# Metrics
def nse(obs, sim):
    return float(1 - np.sum((obs - sim)**2) / np.sum((obs - obs.mean())**2))

def kge(obs, sim):
    r = obs.corr(sim)
    alpha = sim.std() / obs.std()
    beta = sim.mean() / obs.mean()
    return float(1 - np.sqrt((r-1)**2 + (alpha-1)**2 + (beta-1)**2))

def pbias(obs, sim):
    return float(100 * (sim.sum() - obs.sum()) / obs.sum())

metrics = {
    'NSE': round(nse(obs_clean, sim_clean), 3),
    'KGE': round(kge(obs_clean, sim_clean), 3),
    'PBIAS': round(pbias(obs_clean, sim_clean), 1)
}

print(f"\nPerformance Metrics ({model_name}, Uncalibrated):")
for k, v in metrics.items():
    print(f"  {k}: {v}")


In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Time series (top left)
axes[0, 0].plot(obs_clean.index, obs_clean.values, 'b-', label='Observed (USGS)', linewidth=1.2, alpha=0.7)
axes[0, 0].plot(sim_clean.index, sim_clean.values, 'r-', label=f'Simulated ({model_name})', linewidth=1.2, alpha=0.7)
axes[0, 0].set_ylabel('Discharge (m\u00b3/s)')
axes[0, 0].set_title('Streamflow Time Series')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].text(0.02, 0.95, f"NSE: {metrics['NSE']}\nKGE: {metrics['KGE']}\nBias: {metrics['PBIAS']}%",
                transform=axes[0, 0].transAxes, verticalalignment='top',
                bbox=dict(facecolor='white', alpha=0.8), fontsize=9)

# Scatter (top right)
axes[0, 1].scatter(obs_clean, sim_clean, alpha=0.5, s=10)
max_val = max(obs_clean.max(), sim_clean.max())
axes[0, 1].plot([0, max_val], [0, max_val], 'k--', alpha=0.5)
axes[0, 1].set_xlabel('Observed (m\u00b3/s)')
axes[0, 1].set_ylabel('Simulated (m\u00b3/s)')
axes[0, 1].set_title('Observed vs Simulated')
axes[0, 1].grid(True, alpha=0.3)

# Monthly climatology (bottom left)
monthly_obs = obs_clean.groupby(obs_clean.index.month).mean()
monthly_sim = sim_clean.groupby(sim_clean.index.month).mean()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
axes[1, 0].plot(monthly_obs.index, monthly_obs.values, 'b-o', label='Observed', markersize=6)
axes[1, 0].plot(monthly_sim.index, monthly_sim.values, 'r-o', label='Simulated', markersize=6)
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].set_xticklabels(month_names)
axes[1, 0].set_ylabel('Mean Discharge (m\u00b3/s)')
axes[1, 0].set_title('Seasonal Flow Regime')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Flow duration curve (bottom right)
obs_sorted = obs_clean.sort_values(ascending=False)
sim_sorted = sim_clean.sort_values(ascending=False)
obs_ranks = np.arange(1., len(obs_sorted) + 1) / len(obs_sorted) * 100
sim_ranks = np.arange(1., len(sim_sorted) + 1) / len(sim_sorted) * 100
axes[1, 1].semilogy(obs_ranks, obs_sorted, 'b-', label='Observed', linewidth=2)
axes[1, 1].semilogy(sim_ranks, sim_sorted, 'r-', label='Simulated', linewidth=2)
axes[1, 1].set_xlabel('Exceedance Probability (%)')
axes[1, 1].set_ylabel('Discharge (m\u00b3/s)')
axes[1, 1].set_title('Flow Duration Curve')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f'Logan River at Logan \u2014 Distributed {model_name} Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 5b - Model calibration

Calibrate model parameters using ASYNC-DDS to improve model performance. The
calibration optimizes KGE over the calibration period.


In [ ]:
# Step 5b — Run calibration
import shutil

print(f"Starting calibration...")
print(f"Algorithm: {config.optimization.algorithm}")
print(f"Metric: {config.optimization.metric}")
print(f"Iterations: {config.optimization.iterations}")
print(f"Calibration period: {config.domain.calibration_period}")
print(f"Parallel workers: {config.system.num_processes}")
print(f"Target evaluations: {config.optimization.iterations * config.system.num_processes}")

# Remove stale calibration state before optimization
model_name = config.model.hydrological_model.upper()
experiment_id = config.domain.experiment_id
optimization_root = project_dir / "optimization" / model_name
optimization_state_names = [
    f"dds_{experiment_id}",
    f"async_dds_{experiment_id}",
    f"async-dds_{experiment_id}",
    f"asyncdds_{experiment_id}",
]
simulation_state_names = [
    "run_dds",
    "run_async_dds",
    "run_async-dds",
    "run_asyncdds",
]
stale_calibration_paths = [
    *(optimization_root / name for name in optimization_state_names),
    *(project_dir / "simulations" / name for name in simulation_state_names),
]
for stale_path in stale_calibration_paths:
    if stale_path.exists():
        shutil.rmtree(stale_path)
        print(f"Removed stale calibration state: {stale_path}")

results_file = symfluence.managers['optimization'].calibrate_model()
print(f"\nCalibration complete!")
print(f"Results file: {results_file}")

### View Calibration Results

In [ ]:
# Load and display calibration results
import json as _json

cal_results = pd.read_csv(results_file)
cal_results['best_score'] = cal_results['score'].cummax()

print("Calibration Progress:")
print(f"  Best {config.optimization.metric}: {cal_results['best_score'].iloc[-1]:.4f}")
print(f"  Initial {config.optimization.metric}: {cal_results['best_score'].iloc[0]:.4f}")
print(f"  Improvement: {cal_results['best_score'].iloc[-1] - cal_results['best_score'].iloc[0]:.4f}")

# --- Load final evaluation metrics and calibrated simulation ---
opt_dir = Path(results_file).parent
fe_dir = opt_dir / 'final_evaluation'

# Final evaluation JSON (calibration + evaluation period metrics)
fe_metrics = {}
fe_jsons = sorted(opt_dir.glob('*_final_evaluation.json'))
if fe_jsons:
    with open(fe_jsons[0]) as _f:
        fe_data = _json.load(_f)
    fe_metrics['calib'] = {k.replace('Calib_', ''): v for k, v in fe_data.get('calibration_metrics', {}).items()}
    fe_metrics['eval'] = {k.replace('Eval_', ''): v for k, v in fe_data.get('evaluation_metrics', {}).items()}
    print(f"\nCalibration period:  KGE={fe_metrics['calib'].get('KGE', 'N/A'):.3f}  "
          f"NSE={fe_metrics['calib'].get('NSE', 'N/A'):.3f}")
    print(f"Evaluation period:   KGE={fe_metrics['eval'].get('KGE', 'N/A'):.3f}  "
          f"NSE={fe_metrics['eval'].get('NSE', 'N/A'):.3f}")

# Load calibrated simulation from final_evaluation directory
cal_sim = None
if fe_dir.exists():
    # Prefer routed or nexus outlet files before catchment diagnostic CSVs
    csv_paths = sorted(
        fe_dir.glob('*.csv'),
        key=lambda path: (
            0 if path.stem.startswith('nex-') or path.stem.endswith('_output') else 1,
            path.name,
        ),
    )
    for csv_path in csv_paths:
        _df = pd.read_csv(csv_path)
        is_headerless_ngen = False
        if len(_df.columns) == 3:
            try:
                pd.to_datetime(_df.columns[1])
                is_headerless_ngen = True
            except (TypeError, ValueError):
                pass
        if is_headerless_ngen:
            _df = pd.read_csv(
                csv_path,
                header=None,
                names=['index', 'datetime', 'flow'],
            )

        time_col = next(
            (col for col in ['datetime', 'time', 'Time'] if col in _df.columns),
            None,
        )
        flow_col = next(
            (
                col for col in [
                    'streamflow_cms', 'discharge_cms', 'NGEN_discharge_cms',
                    'flow', 'Flow', 'q_cms', 'Q_OUT',
                ]
                if col in _df.columns
            ),
            None,
        )
        if time_col is None or flow_col is None:
            continue

        cal_index = pd.to_datetime(_df[time_col], utc=True, errors='coerce')
        cal_index = cal_index.dt.tz_convert(None)
        cal_values = pd.to_numeric(_df[flow_col], errors='coerce')
        cal_sim = pd.Series(
            cal_values.values,
            index=cal_index,
            name=flow_col,
        ).dropna()
        cal_sim = cal_sim[~cal_sim.index.isna()].sort_index()
        if not cal_sim.empty:
            print(f"Loaded calibrated simulation: {csv_path.name} ({flow_col})")
            break
        cal_sim = None
    if cal_sim is None:
        for nc_path in sorted(fe_dir.glob('*.nc')):
            try:
                import xarray as xr
                _ds = xr.open_dataset(nc_path)
                for var in ['averageRoutedRunoff_mean', 'averageRoutedRunoff', 'scalarTotalRunoff']:
                    if var in _ds:
                        cal_sim = _ds[var].to_series()
                        break
                _ds.close()
            except Exception:
                pass
            if cal_sim is not None:
                break

# Load observed streamflow
obs_dir = project_dir / "data" / "observations" / "streamflow" / "preprocessed"
obs_path = obs_dir / f"{config.domain.name}_streamflow_processed.csv"
cal_obs = None
if obs_path.exists():
    _obs = pd.read_csv(obs_path)
    _obs['datetime'] = pd.to_datetime(_obs['datetime'], utc=True, errors='coerce').dt.tz_convert(None)
    _obs.set_index('datetime', inplace=True)
    cal_obs = _obs['discharge_cms'].resample('D').mean()

# --- Build figure ---
if cal_sim is not None and cal_obs is not None:
    cal_sim_daily = cal_sim.resample('D').mean()
    common = cal_sim_daily.index.intersection(cal_obs.index)
    obs_c = cal_obs.loc[common].dropna()
    sim_c = cal_sim_daily.loc[obs_c.index]
    valid = ~(np.isnan(obs_c.values) | np.isnan(sim_c.values))
    obs_c, sim_c = obs_c[valid], sim_c[valid]

    cal_nse = round(nse(obs_c, sim_c), 3)
    cal_kge = round(kge(obs_c, sim_c), 3)
    cal_pbias = round(pbias(obs_c, sim_c), 1)

    fig, axes = plt.subplots(2, 2, figsize=(16, 11))

    # (0,0) Calibration trajectory
    axes[0, 0].plot(cal_results['iteration'], cal_results['best_score'], 'b-', linewidth=2)
    axes[0, 0].scatter(cal_results['iteration'].iloc[-1], cal_results['best_score'].iloc[-1],
                       color='red', s=80, zorder=5, label=f"Final: {cal_results['best_score'].iloc[-1]:.3f}")
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].set_ylabel(f'Best {config.optimization.metric}')
    axes[0, 0].set_title(f'Calibration Trajectory ({config.optimization.algorithm})')
    axes[0, 0].legend(loc='lower right')
    axes[0, 0].grid(True, alpha=0.3)

    # (0,1) Time series — calibrated
    axes[0, 1].plot(obs_c.index, obs_c.values, 'b-', label='Observed (USGS)', linewidth=1.2, alpha=0.7)
    axes[0, 1].plot(sim_c.index, sim_c.values, 'r-', label=f'Calibrated ({model_name})', linewidth=1.2, alpha=0.7)
    axes[0, 1].set_ylabel('Discharge (m³/s)')
    axes[0, 1].set_title('Calibrated Streamflow')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].text(0.02, 0.95, f"NSE: {cal_nse}\nKGE: {cal_kge}\nBias: {cal_pbias}%",
                    transform=axes[0, 1].transAxes, verticalalignment='top',
                    bbox=dict(facecolor='white', alpha=0.8), fontsize=9)

    # (1,0) Seasonal flow regime
    mo = obs_c.groupby(obs_c.index.month).mean()
    ms = sim_c.groupby(sim_c.index.month).mean()
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    axes[1, 0].plot(mo.index, mo.values, 'b-o', label='Observed', markersize=6)
    axes[1, 0].plot(ms.index, ms.values, 'r-o', label='Calibrated', markersize=6)
    axes[1, 0].set_xticks(range(1, 13))
    axes[1, 0].set_xticklabels(month_names)
    axes[1, 0].set_ylabel('Mean Discharge (m³/s)')
    axes[1, 0].set_title('Seasonal Flow Regime')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # (1,1) Flow duration curve
    obs_s = obs_c.sort_values(ascending=False)
    sim_s = sim_c.sort_values(ascending=False)
    obs_r = np.arange(1., len(obs_s) + 1) / len(obs_s) * 100
    sim_r = np.arange(1., len(sim_s) + 1) / len(sim_s) * 100
    axes[1, 1].semilogy(obs_r, obs_s, 'b-', label='Observed', linewidth=2)
    axes[1, 1].semilogy(sim_r, sim_s, 'r-', label='Calibrated', linewidth=2)
    axes[1, 1].set_xlabel('Exceedance Probability (%)')
    axes[1, 1].set_ylabel('Discharge (m³/s)')
    axes[1, 1].set_title('Flow Duration Curve')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle(f'Logan River at Logan — Calibrated {model_name} Results',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(cal_results['iteration'], cal_results['best_score'], 'b-', linewidth=2)
    ax.set_xlabel('Iteration')
    ax.set_ylabel(f'Best {config.optimization.metric}')
    ax.set_title('Calibration Progress')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    if cal_sim is None:
        print("No calibrated simulation output found in final_evaluation/")
    if cal_obs is None:
        print("No observed streamflow found for comparison")